# 80 — Generate the w3id `.htaccess`

Writes `docs/htaccess.txt`, the `/cdisc/cosmos/.htaccess` to submit to
[perma-id/w3id.org](https://github.com/perma-id/w3id.org). Not a deliverable of
this repo — it is submitted elsewhere — but derived here rather than hand-written,
because one part of it is a function of the overlay ontology: the list of term IRIs
that must resolve to the T-Box rather than to the instance graph. Runs after
`70_generate_qbc.ipynb`; `scripts/htaccess_check.py` then applies the rules to
every w3id IRI the ten deliverables carry.

**The rule the file exists to state: a w3id IRI resolves to a document that has
it as subject.** That is linked-data rule three — look up the IRI, get useful
information about that thing — and it decides everything below:

- A **term** of the overlay ontology (class, property; an enum value carries `#`
  and the server sees the enum) resolves to `cosmos_qbc_v1`. The 44 terms are read
  from the T-Box and written as one alternation, so a term added by a later
  decision changes this file on the next run and is caught by the checker before
  the w3id PR that carries it.
- An **individual** — a qualified concept, a scale node, a use-node, a category
  label-node, a (concept, DEC) pair — resolves to its instance graph. Under `bc/`
  every minted IRI is one (the core's terms are CDISC's, under cdisc.org); under
  `qbc/` everything that is not a term is one.
- An **ontology IRI** and a **version IRI** resolve to the ontology at the pinned
  or the named tag; `sdtm/` has no instance graph (D4), so everything under it is
  the ontology.
- `dss/` gets **no rule**: the graph that describes a recording at its own grain
  is the deferred DSS A-Box (D4), and a redirect to the overlay A-Box would say
  "described" about the deferred half. Unmatched, it falls through to the w3id
  404 — the state D17 records as dangling by design.

**The target is the GitHub Pages site, not the raw file.** `raw.githubusercontent.com`
serves Turtle as `text/plain`; a client that asked for `text/turtle` would be sent
to a document that says it is not Turtle. Pages serves by extension, and the site
is rebuilt from every release tag by `.github/workflows/pages.yml`, which also
derives N-Triples, RDF/XML and JSON-LD from the canonical Turtle — so content
negotiation below is over serializations that exist, and a browser lands on the
version's index page, where per-IRI HTML anchors go later (targets change; IRIs
never do).

## Configuration

In [ ]:
ROOT = ".."

QBC_TBOX = f"{ROOT}/cosmos_qbc_v1.ttl"
TARGET   = f"{ROOT}/docs/htaccess.txt"

VERSION = "0.3.0"                     # the release this file was generated against; named in comments only
SITE    = "https://kerfors.github.io/cosmos-rdf"
REPO    = "https://github.com/kerfors/cosmos-rdf"

# Non-version IRIs resolve through /latest/, which build_pages.py writes as a copy of
# the newest tag. A release therefore needs no change here and no w3id pull request;
# a version IRI still resolves to its own tag through the generic $1 rules below.
LATEST  = f"{SITE}/latest"

# WIDOCO documentation per ontology, written by build_pages.py into every release.
# A browser asking for an ontology or one of its terms lands here rather than on the
# release index. Term anchors are the term local names, which is what WIDOCO writes
# as its entity div ids.
DOC = {
    "cosmos_bc_v1": "doc-cosmos_bc_v1/index-en.html",
    "cosmos_sdtm_v1": "doc-cosmos_sdtm_v1/index-en.html",
    "cosmos_qbc_v1": "doc-cosmos_qbc_v1/index-en.html",
}

QBC_NS = "https://w3id.org/cdisc/cosmos/qbc/"

# Accept media type (regex, as mod_rewrite sees it) -> file extension on the site.
NEGOTIATED = [
    (r"application/n-triples", "nt"),
    (r"application/rdf\+xml", "rdf"),
    (r"application/ld\+json", "jsonld"),
]
BROWSER = [r"text/html", r"application/xhtml\+xml"]   # plus a Mozilla user agent

## The overlay's terms, read from the T-Box

Every subject the overlay ontology declares under `qbc/` that is a class or a
property. Enum values (`…Enum#value`) are excluded: the fragment never reaches the
server, so the enum itself is the term that resolves. The count is asserted so a
change in the ontology is a visible change here.

In [ ]:
from pathlib import Path

from rdflib import Graph, URIRef
from rdflib.namespace import OWL, RDF

tbox = Graph().parse(QBC_TBOX, format="turtle")

TERM_TYPES = (OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.AnnotationProperty, RDF.Property)
terms = sorted({
    str(s)[len(QBC_NS):]
    for term_type in TERM_TYPES
    for s in tbox.subjects(RDF.type, term_type)
    if str(s).startswith(QBC_NS) and "#" not in str(s)
})

unaccounted = sorted({
    str(s)[len(QBC_NS):] for s in tbox.subjects()
    if isinstance(s, URIRef) and str(s).startswith(QBC_NS) and "#" not in str(s)
} - set(terms) - {""})
if unaccounted:
    raise RuntimeError(f"qbc/ subjects in the T-Box that are neither class nor property: {unaccounted}")
if any(not t.replace("_", "").isalnum() for t in terms):
    raise RuntimeError("a term name needs regex escaping; extend the rule writer")

print(f"{len(terms)} overlay terms resolve to the ontology:")
print("   ", ", ".join(terms))

## Write the rules

Order matters and each rule ends the chain (`[L]`): fixed paths first, so a
context or shapes IRI resolves whatever the Accept header; then version IRIs, one
generic rule per graph and per serialization; then the ontology roots and the
overlay's terms; then the instance graphs as the catch-all under `bc/` and `qbc/`;
`sdtm/` last of the segments; the namespace root to the repository. No rule for
`dss/`.

Each negotiated group is one pattern written four times: three `RewriteCond`-gated
rules for N-Triples, RDF/XML and JSON-LD, one browser rule to the version's index
page, and a final unconditioned rule to Turtle that also serves as the default for
clients sending no Accept header.

In [ ]:
V = LATEST


def group(pattern, base, version_capture=False, browser=None, browser_flags="R=303,L"):
    """The rule block for one pattern resolving to one graph file base.

    `browser` overrides where a browser lands - a WIDOCO page rather than the release
    index. It may carry a $1 backreference, in which case the caller must pass the NE
    flag: mod_rewrite percent-encodes a # in a target otherwise, and GitHub answers
    %23 with 400 (measured live 2026-09-04, the defect w3id PR #6643 had to fix)."""
    site = f"{SITE}/v$1" if version_capture else V
    block = []
    for accept, ext in NEGOTIATED:
        block.append(f"RewriteCond %{{HTTP_ACCEPT}} {accept}")
        block.append(f"RewriteRule {pattern} {site}/{base}.{ext} [R=303,L]")
    for accept in BROWSER:
        block.append(f"RewriteCond %{{HTTP_ACCEPT}} {accept} [OR]")
    block.append("RewriteCond %{HTTP_USER_AGENT} ^Mozilla/.*")
    block.append(f"RewriteRule {pattern} {site}/{browser or 'index.html'} [{browser_flags}]")
    block.append(f"RewriteRule {pattern} {site}/{base}.ttl [R=303,L]")
    return block


VERSION_RE = r"([0-9]+\.[0-9]+\.[0-9]+)"
TERMS_RE = "|".join(terms)

out = []
out += Path(f"{ROOT}/docs/htaccess-header.txt").read_text(encoding="utf-8").rstrip("\n").splitlines()
out += ["", "RewriteEngine on", "RewriteBase /cdisc/cosmos", ""]

out += ["# --- Fixed paths: the contexts and the shapes graphs, at the current release.",
        "#     First, so they resolve whatever the Accept header says."]
out.append(f"RewriteRule ^bc/context\.jsonld$   {V}/cosmos_bc_v1.context.jsonld [R=303,L]")
out.append(f"RewriteRule ^sdtm/context\.jsonld$ {V}/cosmos_sdtm_v1.context.jsonld [R=303,L]")
out.append(f"RewriteRule ^bc/shapes$            {V}/cosmos_bc_v1.shapes.ttl [R=303,L]")
out.append(f"RewriteRule ^sdtm/shapes$          {V}/cosmos_sdtm_v1.shapes.ttl [R=303,L]")
out.append(f"RewriteRule ^qbc/shapes$           {V}/cosmos_qbc_v1.shapes.ttl [R=303,L]")
out.append("")

out += ["# --- Version IRIs: <graph>/X.Y.Z resolves to release vX.Y.Z of that graph.",
        "#     One generic block per graph; a release needs no change here."]
for pattern, base in [
    (f"^bc/instances/{VERSION_RE}$", "cosmos_bc_v1.instances"),
    (f"^qbc/instances/{VERSION_RE}$", "cosmos_qbc_v1.instances"),
    (f"^bc/{VERSION_RE}$", "cosmos_bc_v1"),
    (f"^sdtm/{VERSION_RE}$", "cosmos_sdtm_v1"),
    (f"^qbc/{VERSION_RE}$", "cosmos_qbc_v1"),
]:
    out += group(pattern, base, version_capture=True)
    out.append("")

out += ["# --- Ontology IRIs (segment root, with or without the slash) and, for the",
        "#     overlay, its terms: the classes and properties the T-Box declares,",
        f"#     read from cosmos_qbc_v1.ttl by 80_generate_htaccess.ipynb ({len(terms)} at v{VERSION}).",
        "#     A browser gets the WIDOCO description; an RDF client gets the ontology.",
        "#     Three overlay terms carry no rdfs:label and so have no anchor in the page",
        "#     (gen-owl emits a bare declaration for a name declared on more than one",
        "#     class - overlay schema, D19); those land at the top of it."]
out += group("^bc/?$", "cosmos_bc_v1", browser=DOC["cosmos_bc_v1"])
out.append("")
out += group("^qbc/?$", "cosmos_qbc_v1", browser=DOC["cosmos_qbc_v1"])
out.append("")
out += group(f"^qbc/({TERMS_RE})$", "cosmos_qbc_v1",
             browser=f"{DOC['cosmos_qbc_v1']}#$1", browser_flags="R=303,L,NE")
out.append("")

out += ["# --- Every other IRI under bc/ or qbc/ is an individual - a category label-node,",
        "#     a (concept, DEC) pair, a qualified concept, a scale node, a use-node - and",
        "#     resolves to the instance graph that has it as subject."]
out += group("^bc/.+", "cosmos_bc_v1.instances")
out.append("")
out += group("^qbc/.+", "cosmos_qbc_v1.instances")
out.append("")

out += ["# --- sdtm/ has no instance graph (decision D4): the root and anything under it",
        "#     resolve to the ontology. The class and property IRIs of the core rendering",
        "#     are CDISC's own https://www.cdisc.org/cosmos/ strings and are not served here,",
        "#     so there is no per-term rule to write for this segment."]
out += group("^sdtm(/.*)?$", "cosmos_sdtm_v1", browser=DOC["cosmos_sdtm_v1"])
out.append("")

out += ["# --- Namespace root: no single graph describes the whole namespace; the site's",
        "#     release index does. (Not the repository README: a fragment in a redirect",
        "#     target gets percent-encoded by mod_rewrite unless flagged NE, and GitHub",
        "#     answers %23readme with 400 - measured live 2026-09-04.)",
        f"RewriteRule ^$ {SITE}/index.html [R=303,L]",
        "",
        "# dss/ deliberately has no rule (see header): unmatched, it falls through to",
        "# the w3id 404 until the Dataset Specialization layer lands (D4)."]

text = "\n".join(out) + "\n"
Path(TARGET).write_text(text, encoding="utf-8")
rules = sum(1 for line in out if line.startswith("RewriteRule"))
print(f"{TARGET}  {rules} RewriteRule lines, {len(out)} lines")

## Check

`scripts/htaccess_check.py` is the real check — every w3id IRI in the ten
deliverables, under every Accept header the file negotiates. Run it from the repo
root after this notebook. Here only the shape of the file is asserted.

In [ ]:
written = Path(TARGET).read_text(encoding="utf-8").splitlines()
conds = [line for line in written if line.startswith("RewriteCond")]
rules = [line for line in written if line.startswith("RewriteRule")]
if not any(f"^qbc/({TERMS_RE})$" in line for line in rules):
    raise RuntimeError("the term alternation did not make it into the file")
if any("dss/" in line for line in rules):
    raise RuntimeError("a rule matches dss/; the segment is reserved")
if any("raw.githubusercontent.com" in line for line in rules):
    raise RuntimeError("a rule targets the raw file; targets are the Pages site")
if any(f"/v{VERSION}/" in line for line in rules):
    raise RuntimeError("a rule still targets a pinned release; non-version IRIs go through /latest/")
anchored = [line for line in rules if "index-en.html#$1" in line]
if len(anchored) != 1:
    raise RuntimeError(f"expected one term-anchor rule, found {len(anchored)}")
if not anchored[0].rstrip().endswith("[R=303,L,NE]"):
    raise RuntimeError("the term-anchor rule needs the NE flag or the # is percent-encoded")
print(f"ok    {len(rules)} rules, {len(conds)} conditions, terms alternation present, "
      f"no dss/ rule, no raw target, no pinned release, anchor rule carries NE")